# Week 5: Embeddings Deep Dive

**Lab companion to [week_05.md](../agenda/week_05.md)**

In this lab, you will:
1. Understand vector embeddings visually
2. Measure similarity between texts
3. Build a semantic search engine
4. Compare embedding models

In [ ]:
# Setup
from openai import OpenAI
from dotenv import load_dotenv
import numpy as np

load_dotenv()
client = OpenAI()

print("✓ Ready!")

## 1. What Are Embeddings?

Embeddings convert text into numbers (vectors) that capture meaning.

In [ ]:
def get_embedding(text: str) -> list:
    """Get embedding for text."""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

# Get an embedding
text = "Hello, world!"
embedding = get_embedding(text)

print(f"Text: '{text}'")
print(f"Embedding dimensions: {len(embedding)}")
print(f"First 10 values: {embedding[:10]}")

In [ ]:
# Visualize embedding distribution
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(embedding[:100])
plt.title("First 100 dimensions")
plt.xlabel("Dimension")
plt.ylabel("Value")

plt.subplot(1, 2, 2)
plt.hist(embedding, bins=50)
plt.title("Value distribution")
plt.xlabel("Value")
plt.ylabel("Count")

plt.tight_layout()
plt.show()

## 2. Measuring Similarity

Cosine similarity: 1.0 = identical, 0.0 = unrelated, -1.0 = opposite

In [ ]:
def cosine_similarity(a: list, b: list) -> float:
    """Calculate cosine similarity between two vectors."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Test with similar and different sentences
sentences = [
    "I love programming in Python",
    "Python is my favorite programming language",
    "I enjoy coding with Python",
    "The weather is nice today",
    "It's sunny outside"
]

# Get embeddings
embeddings = {s: get_embedding(s) for s in sentences}

# Compare first sentence to all others
base = sentences[0]
print(f"Comparing to: '{base}'\n")

for s in sentences[1:]:
    sim = cosine_similarity(embeddings[base], embeddings[s])
    bar = "█" * int(sim * 20)
    print(f"{sim:.3f} {bar} '{s}'")

In [ ]:
# Similarity matrix
n = len(sentences)
sim_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = cosine_similarity(
            embeddings[sentences[i]], 
            embeddings[sentences[j]]
        )

# Visualize
plt.figure(figsize=(8, 6))
plt.imshow(sim_matrix, cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(label='Similarity')

# Labels
short_labels = [s[:25] + "..." if len(s) > 25 else s for s in sentences]
plt.xticks(range(n), short_labels, rotation=45, ha='right')
plt.yticks(range(n), short_labels)
plt.title("Sentence Similarity Matrix")
plt.tight_layout()
plt.show()

## 3. Semantic Search

Find similar documents using embeddings:

In [ ]:
# Knowledge base
knowledge_base = [
    "Python is a high-level programming language known for its simplicity.",
    "JavaScript is the language of the web, running in browsers.",
    "Machine learning uses algorithms to learn patterns from data.",
    "Deep learning is a subset of ML using neural networks.",
    "APIs allow different software systems to communicate.",
    "REST APIs use HTTP methods like GET, POST, PUT, DELETE.",
    "Version control with Git helps track code changes.",
    "Docker containers package applications with dependencies.",
    "Kubernetes orchestrates container deployment at scale.",
    "SQL databases store data in structured tables."
]

# Create embeddings
kb_embeddings = [get_embedding(text) for text in knowledge_base]

def semantic_search(query: str, top_k: int = 3) -> list:
    """Search knowledge base by semantic similarity."""
    query_emb = get_embedding(query)
    
    # Calculate similarities
    scores = [
        (text, cosine_similarity(query_emb, emb))
        for text, emb in zip(knowledge_base, kb_embeddings)
    ]
    
    # Sort by similarity
    scores.sort(key=lambda x: x[1], reverse=True)
    
    return scores[:top_k]

# Test searches
queries = [
    "What's a good language for beginners?",
    "How do neural networks work?",
    "How to manage code versions?"
]

for query in queries:
    print(f"Query: '{query}'")
    results = semantic_search(query)
    for text, score in results:
        print(f"  {score:.3f}: {text}")
    print()

## 4. Batch Embeddings (Efficient)

Process many texts in one API call:

In [ ]:
def get_embeddings_batch(texts: list) -> list:
    """Get embeddings for multiple texts in one call."""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    return [item.embedding for item in response.data]

# Batch process
texts = [
    "First document",
    "Second document",
    "Third document"
]

embeddings = get_embeddings_batch(texts)
print(f"Got {len(embeddings)} embeddings")
print(f"Each has {len(embeddings[0])} dimensions")

In [ ]:
# Compare speed: batch vs individual
import time

test_texts = [f"This is document number {i}" for i in range(10)]

# Individual calls
start = time.time()
individual = [get_embedding(t) for t in test_texts]
individual_time = time.time() - start

# Batch call
start = time.time()
batch = get_embeddings_batch(test_texts)
batch_time = time.time() - start

print(f"Individual calls: {individual_time:.2f}s")
print(f"Batch call: {batch_time:.2f}s")
print(f"Speedup: {individual_time/batch_time:.1f}x")

## 5. Embedding Dimensions

text-embedding-3 models support dimension reduction:

In [ ]:
text = "Machine learning is fascinating"

# Full dimensions
response_full = client.embeddings.create(
    model="text-embedding-3-small",
    input=text
)
full_emb = response_full.data[0].embedding

# Reduced dimensions
response_small = client.embeddings.create(
    model="text-embedding-3-small",
    input=text,
    dimensions=256  # Reduce from 1536 to 256
)
small_emb = response_small.data[0].embedding

print(f"Full embedding: {len(full_emb)} dimensions")
print(f"Reduced embedding: {len(small_emb)} dimensions")
print(f"Storage reduction: {len(small_emb)/len(full_emb)*100:.1f}%")

## 6. Building a Search Index Class

In [ ]:
class EmbeddingIndex:
    """Simple in-memory embedding search index."""
    
    def __init__(self, model: str = "text-embedding-3-small"):
        self.client = OpenAI()
        self.model = model
        self.documents = []
        self.embeddings = []
    
    def add(self, text: str, metadata: dict = None):
        """Add a document to the index."""
        embedding = self._embed(text)
        self.documents.append({
            'text': text,
            'metadata': metadata or {}
        })
        self.embeddings.append(embedding)
    
    def add_batch(self, texts: list, metadatas: list = None):
        """Add multiple documents efficiently."""
        embeddings = self._embed_batch(texts)
        
        for i, (text, emb) in enumerate(zip(texts, embeddings)):
            meta = metadatas[i] if metadatas else {}
            self.documents.append({'text': text, 'metadata': meta})
            self.embeddings.append(emb)
    
    def search(self, query: str, top_k: int = 5) -> list:
        """Search for similar documents."""
        query_emb = self._embed(query)
        
        scores = []
        for i, emb in enumerate(self.embeddings):
            score = self._similarity(query_emb, emb)
            scores.append((self.documents[i], score))
        
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]
    
    def _embed(self, text: str) -> list:
        response = self.client.embeddings.create(
            model=self.model,
            input=text
        )
        return response.data[0].embedding
    
    def _embed_batch(self, texts: list) -> list:
        response = self.client.embeddings.create(
            model=self.model,
            input=texts
        )
        return [item.embedding for item in response.data]
    
    def _similarity(self, a: list, b: list) -> float:
        a, b = np.array(a), np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
    
    def __len__(self):
        return len(self.documents)

# Create and test
index = EmbeddingIndex()

# Add documents
docs = [
    ("Python is great for data science", {"topic": "programming"}),
    ("JavaScript powers web applications", {"topic": "programming"}),
    ("Machine learning predicts outcomes", {"topic": "AI"}),
    ("Deep learning uses neural networks", {"topic": "AI"}),
    ("REST APIs enable communication", {"topic": "backend"})
]

for text, meta in docs:
    index.add(text, meta)

print(f"Index has {len(index)} documents")

In [ ]:
# Search the index
results = index.search("What language is good for AI?")

print("Search results:")
for doc, score in results:
    print(f"  {score:.3f} [{doc['metadata'].get('topic', 'N/A')}] {doc['text']}")

## 7. Exercise: Build a FAQ Search

In [ ]:
# Exercise: Create a FAQ search system
faqs = [
    {"q": "How do I reset my password?", "a": "Go to settings > security > change password"},
    {"q": "What payment methods do you accept?", "a": "We accept Visa, Mastercard, and PayPal"},
    {"q": "How can I contact support?", "a": "Email support@company.com or call 1-800-123-4567"},
    {"q": "What is your return policy?", "a": "30-day returns on unused items with receipt"},
    {"q": "How long does shipping take?", "a": "Standard shipping is 5-7 business days"}
]

# Build FAQ index
faq_index = EmbeddingIndex()
for faq in faqs:
    faq_index.add(faq["q"], {"answer": faq["a"]})

# Search function
def answer_question(question: str) -> str:
    results = faq_index.search(question, top_k=1)
    if results and results[0][1] > 0.5:  # Threshold
        return results[0][0]['metadata']['answer']
    return "I don't have an answer for that. Please contact support."

# Test
test_questions = [
    "I forgot my login credentials",
    "Can I pay with Apple Pay?",
    "How do I return something?",
    "What's the meaning of life?"  # Should fail threshold
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {answer_question(q)}")
    print()

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

Build on the SemanticSearch class:
1. Add FAQ pairs (question + answer)
2. Embed the questions only
3. On search, find similar questions
4. Return the answer for the matched question

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
class FAQSearch:
    def __init__(self):
        self.faqs = []
        self.embeddings = []
    
    def add_faq(self, question: str, answer: str):
        embedding = get_embedding(question)
        self.faqs.append({"question": question, "answer": answer})
        self.embeddings.append(embedding)
    
    def search(self, query: str) -> dict:
        query_emb = get_embedding(query)
        scores = [cosine_similarity(query_emb, emb) for emb in self.embeddings]
        best_idx = scores.index(max(scores))
        return self.faqs[best_idx]

faq = FAQSearch()
faq.add_faq("How do I reset my password?", "Go to Settings > Security > Reset Password")
faq.add_faq("What are the office hours?", "9 AM to 5 PM, Monday to Friday")
result = faq.search("I forgot my password")
print(result["answer"])
```

</details>

## Summary

You learned:
- ✅ What embeddings are (text → vectors)
- ✅ How to measure similarity (cosine)
- ✅ Building semantic search
- ✅ Batch processing for efficiency
- ✅ Dimension reduction

**Next:** [Week 6 - Production RAG Architecture](week_06_production_rag.ipynb)